# Model Experiments
Compare various baseline models, ensemble models and boosting models.

In [1]:
import pandas as pd
import sys
sys.path.append("..")
from src.models.train import train_and_save_model

df = pd.read_csv("../data/processed/cleaned_tickets.csv")
df.head()

,Customer Email,Product Purchased,category,sentiment,Combined Text,priority,text
0,carrollallison@example.com,gopro hero,technical issue,product setup,i'm having an issue with the gopro hero. pleas...,critical,i'm having an issue with the gopro hero. pleas...
1,clarkeashley@example.com,lg smart tv,technical issue,peripheral compatibility,i'm having an issue with the lg smart tv. plea...,critical,i'm having an issue with the lg smart tv. plea...
2,gonzalestracy@example.com,dell xps,technical issue,network problem,i'm facing a problem with my dell xps. the del...,low,i'm facing a problem with my dell xps. the del...
3,bradleyolson@example.org,microsoft office,billing inquiry,account access,i'm having an issue with the microsoft office....,low,i'm having an issue with the microsoft office....
4,bradleymark@example.com,autodesk autocad,billing inquiry,data loss,i'm having an issue with the autodesk autocad....,low,i'm having an issue with the autodesk autocad....


## Predict Category

In [2]:
from src.features.feature_engineering import get_tfidf_vectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Split and Vectorize
X_train, X_test, y_train, y_test = train_test_split(df["text"], df["category"], test_size=0.2, random_state=42)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

vec = get_tfidf_vectorizer()
X_train_vec = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(class_weight="balanced")
}

for name, model in models.items():
    print(f"--- {name} ---")
    model.fit(X_train_vec, y_train_enc)
    y_pred = model.predict(X_test_vec)
    print(classification_report(y_test_enc, y_pred, target_names=le.classes_, zero_division=0))


--- Logistic Regression ---
                      precision    recall  f1-score   support

     billing inquiry       0.18      0.17      0.17       357
cancellation request       0.15      0.17      0.16       327
     product inquiry       0.20      0.24      0.22       316
      refund request       0.18      0.17      0.18       345
     technical issue       0.22      0.19      0.20       349

            accuracy                           0.19      1694
           macro avg       0.19      0.19      0.19      1694
        weighted avg       0.19      0.19      0.19      1694

--- Naive Bayes ---
                      precision    recall  f1-score   support

     billing inquiry       0.18      0.10      0.13       357
cancellation request       0.16      0.20      0.18       327
     product inquiry       0.17      0.17      0.17       316
      refund request       0.20      0.23      0.21       345
     technical issue       0.18      0.19      0.19       349

            accur

## Ensemble Learning

In [4]:
from sklearn.ensemble import VotingClassifier

estimators = [(name, model) for name, model in models.items()]
voting = VotingClassifier(estimators=estimators, voting="hard")
voting.fit(X_train_vec, y_train_enc)
print("Voting Classifier Report:")
print(classification_report(y_test_enc, voting.predict(X_test_vec), target_names=le.classes_, zero_division=0))

Voting Classifier Report:
                      precision    recall  f1-score   support

     billing inquiry       0.18      0.17      0.18       357
cancellation request       0.15      0.18      0.17       327
     product inquiry       0.21      0.23      0.22       316
      refund request       0.20      0.19      0.19       345
     technical issue       0.21      0.17      0.19       349

            accuracy                           0.19      1694
           macro avg       0.19      0.19      0.19      1694
        weighted avg       0.19      0.19      0.19      1694

